# Bengali LLM Benchmark: BongLLaMA-3B-Instruct

Largest Bengali instruct model (3B params, BanglaLLM)

**Evaluation**: Zero-shot + 5-shot on full test set (3,553 samples)
**Metrics**: Macro F1 (Type, Target, Severity), CVR, Parse Rate, Latency


In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
import os
import json
import time
import glob
import re
import gc
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = '/kaggle/working'
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.9 MB/s eta 0:00:00
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


---
## 1. Load Test Data


In [2]:
# ── Load Data (same split as NB03/NB04) ──
data_files = glob.glob('/kaggle/input/**/train.json', recursive=True)
test_files = glob.glob('/kaggle/input/**/test.json', recursive=True)

if test_files:
    df_test = pd.read_json(test_files[0])
    print(f'Loaded pre-split test set: {test_files[0]} ({len(df_test)} samples)')
elif data_files:
    print('Only train.json found. Splitting to match NB03/NB04...')
    df_all = pd.read_json(data_files[0])
    df_train_split, df_temp = train_test_split(
        df_all, test_size=0.2, random_state=42,
        stratify=df_all['type_of_hate']
    )
    df_dev, df_test = train_test_split(
        df_temp, test_size=0.5, random_state=42,
        stratify=df_temp['type_of_hate']
    )
    print(f'  Train: {len(df_train_split)}, Val: {len(df_dev)}, Test: {len(df_test)}')
else:
    raise FileNotFoundError('No data found!')

# ── Remap labels ──
_LABEL_REMAP = {'Profane': 'Abusive', 'Sexism': 'Gender Hate'}
df_test['type_of_hate'] = df_test['type_of_hate'].fillna('None').astype(str).str.strip().replace(_LABEL_REMAP)
df_test['target_of_hate'] = df_test['target_of_hate'].fillna('None').astype(str).str.strip()
df_test['severity_of_hate'] = df_test['severity_of_hate'].astype(str).str.strip()

# Use FULL test set (3,553 samples)
df_eval = df_test.reset_index(drop=True)
print(f'\nEvaluation samples: {len(df_eval)} (FULL test set)')
print(f'Type distribution:\n{df_eval["type_of_hate"].value_counts().to_string()}')


Only train.json found. Splitting to match NB03/NB04...
  Train: 28417, Val: 3552, Test: 3553

Evaluation samples: 3553 (FULL test set)
Type distribution:
type_of_hate
None              1996
Abusive           1055
Political Hate     423
Religious Hate      67
Gender Hate         12


---
## 2. Shared Utilities


In [3]:
# ── Constants & Normalization Maps ──
VALID_TYPES = {'None', 'Abusive', 'Political Hate', 'Religious Hate', 'Gender Hate'}
VALID_TARGETS = {'None', 'Individual', 'Organization', 'Community', 'Society'}
VALID_SEVERITIES = {'Little to None', 'Mild', 'Severe'}

TYPE_MAP = {k.lower(): k for k in VALID_TYPES}
TYPE_MAP.update({
    'profane': 'Abusive',
    'sexism': 'Gender Hate',
    'hate': 'Abusive',
    'political': 'Political Hate',
    'religious': 'Religious Hate',
    'gender': 'Gender Hate'
})

TARGET_MAP = {k.lower(): k for k in VALID_TARGETS}
TARGET_MAP.update({
    'individual': 'Individual',
    'organization': 'Organization',
    'community': 'Community',
    'society': 'Society'
})

SEV_MAP = {k.lower(): k for k in VALID_SEVERITIES}
SEV_MAP.update({
    'little to none': 'Little to None',
    'none': 'Little to None',
    'mild': 'Mild',
    'severe': 'Severe',
    'medium': 'Mild',
    'low': 'Little to None',
    'high': 'Severe'
})

def check_consistency_violation(type_pred, target_pred, sev_pred):
    """Returns True if prediction violates consistency rules."""
    if type_pred == 'None':
        if target_pred != 'None' or sev_pred != 'Little to None':
            return True
    return False

def extract_labels_from_json(text):
    """Extract structured labels with primary JSON parser + fallback."""
    # 1. Primary: JSON parse
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            parsed = json.loads(match.group(0))
            return (
                parsed.get('type_of_hate', 'None'),
                parsed.get('target_of_hate', 'None'),
                parsed.get('severity_of_hate', 'Little to None')
            ), True
    except:
        pass
    
    # 2. Fallback: line-by-line key extraction
    try:
        t_val, trg_val, s_val = None, None, None
        for line in text.splitlines():
            line_c = line.strip().replace('"', '').replace("'", '')
            if 'type_of_hate' in line_c:
                t_val = line_c.split(':')[-1].split('=')[-1].strip(' ,{}')
            if 'target_of_hate' in line_c:
                trg_val = line_c.split(':')[-1].split('=')[-1].strip(' ,{}')
            if 'severity_of_hate' in line_c:
                s_val = line_c.split(':')[-1].split('=')[-1].strip(' ,{}')
        if t_val or trg_val or s_val:
            return (
                t_val if t_val else 'None',
                trg_val if trg_val else 'None',
                s_val if s_val else 'Little to None'
            ), True
    except:
        pass
        
    return ('None', 'None', 'Little to None'), False

def normalize_prediction(t, trg, s):
    """Normalize casing and aliases to canonical taxonomy."""
    t_str = str(t).strip().lower()
    trg_str = str(trg).strip().lower()
    s_str = str(s).strip().lower()
    
    t_norm = TYPE_MAP.get(t_str, 'None')
    trg_norm = TARGET_MAP.get(trg_str, 'None')
    s_norm = SEV_MAP.get(s_str, 'Little to None')
    return t_norm, trg_norm, s_norm

def build_zero_shot_prompt(comment):
    return f"""নিচের মন্তব্যটি বিশ্লেষণ করুন এবং এর 'Hate Type', 'Target', এবং 'Severity' নির্ধারণ করুন।

অপশনসমূহ:
Hate Type: None, Abusive, Political Hate, Religious Hate, Gender Hate
Target: None, Individual, Organization, Community, Society
Severity: Little to None, Mild, Severe

মন্তব্য: "{comment}"

শুধুমাত্র নিচের JSON ফরম্যাটে উত্তর দিন:
{{"type_of_hate": "...", "target_of_hate": "...", "severity_of_hate": "..."}}"""

def build_few_shot_prompt(comment):
    return f"""নিচের মন্তব্যটি বিশ্লেষণ করুন এবং এর 'Hate Type', 'Target', এবং 'Severity' নির্ধারণ করুন।

অপশনসমূহ:
Hate Type: None, Abusive, Political Hate, Religious Hate, Gender Hate
Target: None, Individual, Organization, Community, Society
Severity: Little to None, Mild, Severe

এখানে কিছু উদাহরণ দেওয়া হলো:

মন্তব্য: "মেসি বোলে কথা মেসি মেসি মেসি এবং মেসি কিছু বলতে হবে না"
উত্তর: {{"type_of_hate": "None", "target_of_hate": "None", "severity_of_hate": "Little to None"}}

মন্তব্য: "সবাই বয়কট করুন সময় টিভি"
উত্তর: {{"type_of_hate": "Abusive", "target_of_hate": "Organization", "severity_of_hate": "Mild"}}

মন্তব্য: "একটা পণ্য না ভাই কোন পণ্যর দাম কম বলব এই সরকার কি আর বাজার নিয়ন্ত্রণ করতে পারবে না ডাকাতি করবে"
উত্তর: {{"type_of_hate": "Political Hate", "target_of_hate": "Society", "severity_of_hate": "Mild"}}

মন্তব্য: "সরকারের উচিত বাংলাদেশেও মন্দির ভেঙে মসজিদ করা"
উত্তর: {{"type_of_hate": "Religious Hate", "target_of_hate": "Organization", "severity_of_hate": "Little to None"}}

মন্তব্য: "আন্দোলনটা ছিল পতিতা মহিলাদের জন্য"
উত্তর: {{"type_of_hate": "Gender Hate", "target_of_hate": "Society", "severity_of_hate": "Severe"}}

এখন এই মন্তব্যটি বিশ্লেষণ করুন:
মন্তব্য: "{comment}"

শুধুমাত্র নিচের JSON ফরম্যাটে উত্তর দিন:
{{"type_of_hate": "...", "target_of_hate": "...", "severity_of_hate": "..."}}"""

def run_benchmark(model, tokenizer, model_name, prompt_fn, max_new_tokens=64):
    """Run benchmark on a loaded model with robust generation and memory management."""
    preds_type, preds_target, preds_sev = [], [], []
    violations = 0
    parse_successes = 0
    start_time = time.time()

    # Determine safe pad token id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    if pad_id is None:
        pad_id = 0

    for idx, (_, row) in enumerate(tqdm(df_eval.iterrows(), total=len(df_eval), desc=f'{model_name}')):
        prompt = prompt_fn(row['comment'])
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=pad_id
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        (t, trg, s), parsed = extract_labels_from_json(response)
        t, trg, s = normalize_prediction(t, trg, s)

        if parsed:
            parse_successes += 1
        if check_consistency_violation(t, trg, s):
            violations += 1

        preds_type.append(t)
        preds_target.append(trg)
        preds_sev.append(s)

        # Clear VRAM cache every 500 samples to prevent fragmentation
        if (idx + 1) % 500 == 0:
            torch.cuda.empty_cache()

    elapsed = time.time() - start_time
    latency = elapsed / len(df_eval)
    parse_rate = parse_successes / len(df_eval) * 100

    return preds_type, preds_target, preds_sev, violations, latency, parse_rate

print('Utilities loaded.')


Utilities loaded.


---
## 3. Benchmark: BongLLaMA-3B-Instruct
Zero-shot evaluation first, then 5-shot. Results saved after each pass.


In [4]:
MODEL_ID = "BanglaLLM/BanglaLLama-3.2-3b-bangla-alpaca-orca-instruct-v0.0.1"
print(f"Loading {MODEL_ID}...")
tokenizer_bong = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_bong = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
model_bong.eval()

# ── PASS 1: Zero-Shot ──
print(f"\n============================================================")
print(f"  ZERO-SHOT EVALUATION: BongLLaMA-3B-Instruct")
print(f"============================================================")
pt_0, ptr_0, ps_0, v_0, lat_0, pr_0 = run_benchmark(
    model_bong, tokenizer_bong, "BongLLaMA-3B-Instruct (0-shot)", build_zero_shot_prompt
)

# Save zero-shot results immediately (safety: if 5-shot times out)
true_type = df_eval['type_of_hate'].tolist()
true_target = df_eval['target_of_hate'].tolist()
true_sev = df_eval['severity_of_hate'].tolist()

def calc_metrics(p_type, p_target, p_sev, viols, latency, parse_rate, name):
    type_f1 = f1_score(true_type, p_type, average='macro', zero_division=0)
    target_f1 = f1_score(true_target, p_target, average='macro', zero_division=0)
    sev_f1 = f1_score(true_sev, p_sev, average='macro', zero_division=0)
    avg_f1 = (type_f1 + target_f1 + sev_f1) / 3
    cvr = viols / len(p_type) * 100
    print(f"  Type F1:       {type_f1:.4f}")
    print(f"  Target F1:     {target_f1:.4f}")
    print(f"  Severity F1:   {sev_f1:.4f}")
    print(f"  Avg Macro F1:  {avg_f1:.4f}")
    print(f"  CVR:           {cvr:.2f}% ({viols}/{len(p_type)})")
    print(f"  Parse Rate:    {parse_rate:.1f}%")
    print(f"  Latency:       {latency:.2f}s/sample")
    return {
        'model': name, 'type_f1': round(type_f1, 4), 'target_f1': round(target_f1, 4),
        'sev_f1': round(sev_f1, 4), 'avg_f1': round(avg_f1, 4), 'cvr_pct': round(cvr, 2),
        'violations': viols, 'total_samples': len(p_type),
        'latency_s': round(latency, 3), 'parse_rate_pct': round(parse_rate, 1)
    }

r_0shot = calc_metrics(pt_0, ptr_0, ps_0, v_0, lat_0, pr_0, "BongLLaMA-3B-Instruct (0-shot)")
with open(os.path.join(RESULTS_DIR, 'bong_0shot_results.json'), 'w') as f:
    json.dump(r_0shot, f, indent=2)
print(f"\n✅ Zero-shot results saved.")

# ── PASS 2: 5-Shot ──
print(f"\n============================================================")
print(f"  5-SHOT EVALUATION: BongLLaMA-3B-Instruct")
print(f"============================================================")
pt_5, ptr_5, ps_5, v_5, lat_5, pr_5 = run_benchmark(
    model_bong, tokenizer_bong, "BongLLaMA-3B-Instruct (5-shot)", build_few_shot_prompt
)

r_5shot = calc_metrics(pt_5, ptr_5, ps_5, v_5, lat_5, pr_5, "BongLLaMA-3B-Instruct (5-shot)")
with open(os.path.join(RESULTS_DIR, 'bong_5shot_results.json'), 'w') as f:
    json.dump(r_5shot, f, indent=2)
print(f"\n✅ 5-shot results saved.")

# Free VRAM
del model_bong, tokenizer_bong
gc.collect()
torch.cuda.empty_cache()


Loading BanglaLLM/BanglaLLama-3.2-3b-bangla-alpaca-orca-instruct-v0.0.1...


config.json:   0%|          | 0.00/917 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

...


  ZERO-SHOT EVALUATION: BongLLaMA-3B-Instruct


BongLLaMA-3B-Instruct (0-shot):   0%|          | 0/3553 [00:00<?, ?it/s]

  Type F1:       0.1439
  Target F1:     0.1500
  Severity F1:   0.2631
  Avg Macro F1:  0.1856
  CVR:           0.00% (0/3553)
  Parse Rate:    0.0%
  Latency:       0.22s/sample

✅ Zero-shot results saved.

  5-SHOT EVALUATION: BongLLaMA-3B-Instruct


BongLLaMA-3B-Instruct (5-shot):   0%|          | 0/3553 [00:00<?, ?it/s]

  Type F1:       0.1439
  Target F1:     0.1500
  Severity F1:   0.0844
  Avg Macro F1:  0.1261
  CVR:           100.00% (3553/3553)
  Parse Rate:    100.0%
  Latency:       1.28s/sample

✅ 5-shot results saved.


---
## 4. Comparison with Our Model


In [5]:
# ── Final Comparison ──
our_result = {
    'model': 'Ours (exp4, 110M)',
    'type_f1': 0.4994, 'target_f1': 0.5598, 'sev_f1': 0.6136,
    'avg_f1': 0.5576, 'cvr_pct': 0.03, 'violations': 1,
    'total_samples': 3553, 'latency_s': 0.015, 'parse_rate_pct': 100.0
}

# Load saved results
with open(os.path.join(RESULTS_DIR, 'bong_0shot_results.json'), 'r') as f:
    r_0shot = json.load(f)

try:
    with open(os.path.join(RESULTS_DIR, 'bong_5shot_results.json'), 'r') as f:
        r_5shot = json.load(f)
    has_5shot = True
except:
    has_5shot = False

# Print final comparison table
print(f"\n\n==========================================================================================")
print(f"  FINAL COMPARISON: BongLLaMA-3B-Instruct vs Our Model")
print(f"==========================================================================================")
print(f"  {'Model':<35} {'Params':>7} {'Avg F1':>8} {'CVR':>8} {'Parse%':>8} {'Lat(s)':>8}")
print(f"  {'-'*35} {'-'*7} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {r_0shot['model']:<35} {'1-3B':>7} {r_0shot['avg_f1']:>8.4f} {r_0shot['cvr_pct']:>7.2f}% {r_0shot['parse_rate_pct']:>7.1f}% {r_0shot['latency_s']:>7.3f}s")
if has_5shot:
    print(f"  {r_5shot['model']:<35} {'1-3B':>7} {r_5shot['avg_f1']:>8.4f} {r_5shot['cvr_pct']:>7.2f}% {r_5shot['parse_rate_pct']:>7.1f}% {r_5shot['latency_s']:>7.3f}s")
print(f"  {'-'*35} {'-'*7} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'Ours (exp4, 110M)':<35} {'110M':>7} {our_result['avg_f1']:>8.4f} {our_result['cvr_pct']:>7.2f}% {our_result['parse_rate_pct']:>7.1f}% {our_result['latency_s']:>7.3f}s")
print(f"==========================================================================================")

# Save all results together
all_results = [r_0shot]
if has_5shot:
    all_results.append(r_5shot)
all_results.append(our_result)
with open(os.path.join(RESULTS_DIR, 'bong_full_comparison.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n✅ All results saved to {RESULTS_DIR}/bong_full_comparison.json")




  FINAL COMPARISON: BongLLaMA-3B-Instruct vs Our Model
  Model                                Params   Avg F1      CVR   Parse%   Lat(s)
  ----------------------------------- ------- -------- -------- -------- --------
  BongLLaMA-3B-Instruct (0-shot)         1-3B   0.1856    0.00%     0.0%   0.218s
  BongLLaMA-3B-Instruct (5-shot)         1-3B   0.1261  100.00%   100.0%   1.277s
  ----------------------------------- ------- -------- -------- -------- --------
  Ours (exp4, 110M)                      110M   0.5576    0.03%   100.0%   0.015s

✅ All results saved to /kaggle/working/results/bong_full_comparison.json
